# 03 — Text Preprocessing & Index-Based Vocabulary Encoding

This notebook presents the core algorithms for text normalization, vocabulary compilation, index-based token encoding, and vectorization pipelines from first principles.

## 1. Why Text Representation Matters

Machine learning algorithms and neural networks operate on continuous or discrete numerical vectors, not raw strings.

To ingest textual data, we construct an **Index-Based Encoding Pipeline**:
1. **Text Normalization:** Standardizing case, stripping typographic noise, removing punctuation, and collapsing whitespace.
2. **Vocabulary Compilation:** Extracting the unique vocabulary $V$ from a corpus and mapping each word to a unique integer index:
   $$w \mapsto 	ext{index}(w) \in \{0, 1, \dots, |V|-1\}$$
3. **Vectorization:** Converting strings into integer token arrays, reserving special tokens (`<PAD>`, `<UNK>`) for variable lengths and out-of-vocabulary terms.

In [ ]:
import json
import re
import string
from pathlib import Path

CORPUS_PATH = Path("../data/sample_articles_corpus.json")
assert CORPUS_PATH.is_file(), f"Corpus missing at {CORPUS_PATH}"
print(f"Verified text corpus: {CORPUS_PATH.resolve()}")

## 2. Text Normalization Pipeline

We implement modular normalization stages:
- Lowercasing: Mapping all characters to lowercase.
- Punctuation Removal: Removing punctuation symbols using translation tables and regex.
- Whitespace Normalization: Collapsing multiple whitespace, tabs, and line breaks into single spaces.

In [ ]:
def text_normalize(text: str) -> str:
    """Normalize text by lowercasing, stripping punctuation, and cleaning whitespace."""
    # 1. Lowercase
    normalized = text.lower()

    # 2. Strip punctuation using str.maketrans
    translator = str.maketrans("", "", string.punctuation)
    normalized = normalized.translate(translator)

    # 3. Collapse multiple whitespace characters into single space
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return normalized


# Verification with noisy input strings
test_inputs = [
    "Hello, we are learning artificial intelligence.",
    "tHIs iS a pRObLEm iN TexT rEPreSEnTAtion",
    "#science?! <artificial intelligence> #deep learning!!!",
    "'#ArTiFIciAL. In?>TeLLi!g@ENce'",
]

print("--- Normalization Verification ---")
for raw in test_inputs:
    print(f"Raw:   {raw}")
    print(f"Clean: {text_normalize(raw)}\n")

## 3. Vocabulary Compilation from a Corpus

Given a collection of paragraphs (corpus), we compile the set of unique tokens and build bidirectional lookup dictionaries:
- `token_to_id`: Maps word string $	o$ integer ID
- `id_to_token`: Maps integer ID $	o$ word string
- Special tokens:
  - `<PAD>`: Index $0$ (for batch padding)
  - `<UNK>`: Index $1$ (for out-of-vocabulary tokens)

In [ ]:
class Vocabulary:
    def __init__(self, pad_token: str = "<PAD>", unk_token: str = "<UNK>"):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.token_to_id: dict[str, int] = {pad_token: 0, unk_token: 1}
        self.id_to_token: dict[int, str] = {0: pad_token, 1: unk_token}

    def build_from_corpus(self, corpus: list[str]) -> None:
        """Extract unique vocabulary words from a normalized corpus."""
        for document in corpus:
            clean_doc = text_normalize(document)
            tokens = clean_doc.split()
            for token in tokens:
                if token not in self.token_to_id:
                    new_id = len(self.token_to_id)
                    self.token_to_id[token] = new_id
                    self.id_to_token[new_id] = token

    def __len__(self) -> int:
        return len(self.token_to_id)

    def get_id(self, token: str) -> int:
        return self.token_to_id.get(token, self.token_to_id[self.unk_token])

    def get_token(self, token_id: int) -> str:
        return self.id_to_token.get(token_id, self.unk_token)


# Load documents from sample articles
with CORPUS_PATH.open(encoding="utf-8") as f:
    articles = json.load(f)

corpus_documents = [f"{a['title']}. {a['abstract']} {a['body']}" for a in articles]
vocab = Vocabulary()
vocab.build_from_corpus(corpus_documents)

print(f"Total compiled vocabulary size: {len(vocab)}")
print("First 20 vocabulary entries:")
for idx in range(min(20, len(vocab))):
    print(f"  ID {idx:3d} -> '{vocab.get_token(idx)}'")

## 4. Sequence Vectorization & Padding

We implement `vectorize()` to map any text string to an integer array with `<UNK>` fallback, and support optional truncation/padding to a fixed sequence length $L$.

In [ ]:
def vectorize(
    text: str,
    vocabulary: Vocabulary,
    max_length: int | None = None,
    pad: bool = False,
) -> list[int]:
    """Vectorize normalized text into integer token IDs."""
    clean_text = text_normalize(text)
    tokens = clean_text.split()

    # Map tokens to IDs
    encoded = [vocabulary.get_id(t) for t in tokens]

    if max_length is not None:
        if len(encoded) > max_length:
            encoded = encoded[:max_length]
        elif pad and len(encoded) < max_length:
            pad_id = vocabulary.token_to_id[vocabulary.pad_token]
            encoded.extend([pad_id] * (max_length - len(encoded)))

    return encoded


# Test vectorization with known and out-of-vocabulary words
sample_query = "Modern machine learning architectures rely heavily on clean data pipelines."
vec = vectorize(sample_query, vocab, max_length=15, pad=True)
decoded = [vocab.get_token(i) for i in vec]

print(f"Query:   {sample_query}")
print(f"Encoded: {vec}")
print(f"Decoded: {decoded}")

## 5. Bag-of-Words and Term Frequency Vectors

Beyond sequential token IDs, foundational NLP relies on fixed-dimensional vector representations:
- **Bag-of-Words (BoW):** Vector $\mathbf{x} \in \mathbb{R}^{|V|}$ where $x_j$ is the frequency count of word $j$.
- **Term Frequency (TF):** $	ext{TF}(w) = rac{	ext{Count}(w)}{\sum_k 	ext{Count}(k)}$

In [ ]:
def compute_term_frequencies(text: str, vocabulary: Vocabulary) -> dict[str, float]:
    """Compute normalized term frequency vector for a text string."""
    clean_text = text_normalize(text)
    tokens = clean_text.split()
    if not tokens:
        return {}

    counts: dict[str, int] = {}
    for t in tokens:
        counts[t] = counts.get(t, 0) + 1

    total = len(tokens)
    return {word: count / total for word, count in sorted(counts.items())}


sample_tf = compute_term_frequencies(articles[0]["abstract"], vocab)
print("Top 10 Term Frequencies in Article 1 Abstract:")
for word, tf in list(sample_tf.items())[:10]:
    print(f"  '{word}': {tf:.4f}")